In [38]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os
import re
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.preprocessing import StandardScaler

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging

In [39]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-small'
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    weight_decay = 0.01
    task = 'regression'

In [40]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [41]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [42]:
path = kagglehub.competition_download(cfg.competition)

In [43]:
train_df = pd.read_csv(f'{path}/train.csv')

In [44]:
train_df['word_count'] = train_df['full_text'].str.split(' ').str.len()

train_df['num_sentences'] = train_df['full_text'].apply(lambda x: len([s for s in re.split(r'[.!?]+', str(x)) if s.strip()]))

train_df['sentence_length'] = train_df[['num_sentences', 'word_count']].apply(lambda row: row['word_count']/row['num_sentences'], axis=1)

train_df['num_para'] = train_df['full_text'].apply(lambda x: len([p for p in str(x).split('\n') if p.strip()]))

train_df['para_length'] = train_df [['num_para', 'word_count']].apply(lambda row: row['word_count']/row['num_para'], axis=1)

train_df['long_essay'] = (train_df['word_count']>2500).astype(int)

In [45]:
feature_cols = ['word_count', 'num_sentences', 'sentence_length', 'num_para', 'para_length', 'long_essay']

In [46]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [47]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [48]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [49]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        self.features = data[feature_cols]
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'features': torch.tensor(self.features.iloc[idx].values, dtype=torch.float)
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.float)
        return item


In [50]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [51]:
class EssayModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        #MLP NN
        self.classifier = nn.Sequential(
            nn.Linear(self.model.config.hidden_size + num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)

        )

    def forward(self, input_ids, attention_mask, features):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)

        #Add feature engineered outputs
        combined_output = torch.cat([mp_output, features], dim=1)
        logits = self.classifier(combined_output)
        return logits


In [52]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
#For CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [53]:
#Global Debug variable for testing
DEBUG = True

In [ ]:
#Store overall best QWK:
fold_QWK = []

#CV Pipeline
for fold in range(5):
    print(f"--------FOLD {fold}--------")

    train_fold = train_df[train_df['fold']!=fold] #Train data are the not current folds
    valid_fold = train_df[train_df['fold']==fold] #Validation is the current fold

    #Create samples for quick testing
    if DEBUG:
        train_df_used = train_fold.sample(frac=0.01, random_state=42)
        valid_df_used = valid_fold #.sample(frac=0.01, random_state=42)
    else:
        train_df_used = train_fold
        valid_df_used = valid_fold
    
    #Scale engineered features
    train_df_used = train_df_used.copy()
    valid_df_used = valid_df_used.copy()
    scaler = StandardScaler()
    train_df_used[feature_cols] = scaler.fit_transform(train_df_used[feature_cols])
    valid_df_used[feature_cols] = scaler.transform(valid_df_used[feature_cols])
    joblib.dump(scaler, f'scaler_fold_{fold}.pkl')
    
    #Create Dataset and DataLoader objects
    train_dataset = EssayDataset(train_df_used, tokenizer)
    valid_dataset = EssayDataset(valid_df_used, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    #Create model object
    logging.set_verbosity_error()
    deberta_model = EssayModel(num_features=len(feature_cols)).to(device).float()
    
    #Create Optimizer
    optimizer = torch.optim.AdamW(
        deberta_model.parameters(), 
        lr=cfg.learning_rate,
        eps=1e-6,
        weight_decay=cfg.weight_decay
        )
   
    #Create Scheduler for changing LR
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    #Define Loss Function
    loss_fn = nn.SmoothL1Loss()

    #Store best QWK, start at a negative value
    best_kappa = -999

    #Early stopping
    patience = 5
    patience_counter = 0

    #Store overall training loss
    overall_train_loss = 0
    overall_valid_loss = 0

    #Train Model Pipeline
    for epoch in range(cfg.epochs):
        print(f'----EPOCH {epoch+1}/{cfg.epochs}----')

        #Train
        deberta_model.train()
        train_loss = 0; train_correct = 0; train_total = 0

        for batch in train_loader:
            optimizer.zero_grad()

            labels = batch['labels'].to(device)

            logits = deberta_model(
                batch['input_ids'].to(device), 
                batch['attention_mask'].to(device),
                batch['features'].to(device)
            ).squeeze(-1)
            
            loss = loss_fn(logits, labels)
            preds = torch.round(logits).clamp(0, 5).long()

            labels_int = labels.long()
            train_correct += (preds == labels_int).sum().item()
            train_total += labels.size(0)
            train_loss += loss.item()
            overall_train_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                deberta_model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = train_correct / train_total


        #Validation
        deberta_model.eval()
        valid_loss = 0; valid_correct = 0; valid_total = 0
        all_labels = []; all_preds = []

        with torch.no_grad():
            for batch in valid_loader:
                labels = batch['labels'].to(device)

                logits = deberta_model(
                    batch['input_ids'].to(device), 
                    batch['attention_mask'].to(device),
                    batch['features'].to(device)
                ).squeeze(-1)
                
                loss = loss_fn(logits, labels)
                valid_loss += loss.item()
                overall_valid_loss += loss.item()
                preds = torch.round(logits).clamp(0, 5).long()
                labels_int = labels.long()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels_int.cpu().numpy())
                valid_correct += (preds==labels_int).sum().item()
                valid_total += labels.size(0)

        avg_valid_loss = valid_loss/len(valid_loader)
        valid_accuracy = valid_correct/valid_total

        kappa = get_score(all_labels, all_preds)
        if kappa > best_kappa:
            best_kappa = kappa
            patience_counter = 0
            torch.save(
                deberta_model.state_dict(),
                f'best_deberta_model_fold_{fold}.pth'
            )
            print(f'Saved new-best model: QWK {best_kappa:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at fold {fold} and epoch {epoch+1}')
                break

        
        #Summary
        print(f'Train Loss: {avg_train_loss:.4f}')
        print(f'Train Accuracy: {train_accuracy:.4f}')
        print(f'Valid Loss: {avg_valid_loss:.4f}')
        print(f'Valid Accuracy: {valid_accuracy:.4f}')
        print(f'Current Epoch QWK: {kappa:.4f}')
        print(f"Overall Fold's Best QWK: {best_kappa:.4f}")
        print(f'LR: {scheduler.get_last_lr()[0]:.2e}\n')
    
    #Store best QWK for that fold
    fold_QWK.append(best_kappa)
    print(f"Fold {fold}'s best QWK: {best_kappa}")
    
    #Print overall loss
    epochs_ran = epoch + 1
    print('Overall average training loss:', overall_train_loss/(epochs_ran*len(train_loader)))
    print('Overall average validation loss:', overall_valid_loss/(epochs_ran*len(valid_loader)))

    #Store fold's last weights
    torch.save(deberta_model.state_dict(),
               f'last_deberta_model_fold_{fold}.pth')


#Mean QWK Across all folds
print('---------QWK SUMMARY---------')
print('Mean QWK Across 5 Folds:', np.mean(fold_QWK))
print(f'Best QWK: {np.max(fold_QWK)} | Fold: {np.argmax(fold_QWK)}')

--------FOLD 0--------


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

----EPOCH 1/10----


In [ ]:
#Save configs to json
import json
config = {
    'competition': cfg.competition,
    'checkpoint': cfg.checkpoint,
    'batch_size': cfg.batch_size,
    'max_length': cfg.max_length,
    'learning_rate': cfg.learning_rate,
    'weight_decay': 0.01,
    'epochs': cfg.epochs,
    'num_labels': 1,
    'task': cfg.task,
    'best_fold': int(np.argmax(fold_QWK))
}
with open('config.json', 'w') as f:
    json.dump(config, f, indent=4)